In [1]:
import os
import numpy as np
import ase
from ase import Atoms
from ase import io
from ase.io.espresso import write_espresso_in as write_in
from ase.io.espresso import read_espresso_out as read_out
from ase.io.lammpsdata import write_lammps_data
from ase.io.lammpsdata import read_lammps_data
from ase.io.lammpsrun import read_lammps_dump_text
from ase.io import write
from ase.io import read
from ase.calculators.espresso import Espresso
from ase.build import make_supercell
from ase.build import surface
from ase.build import sort
from ase.constraints import FixAtoms

In [2]:
outdir="./"
extxyz_path = "./../bulksupercell/bulk_Pt_supercell.extxyz" 
pseudo_dir = "./../pseudo"
pseudopotentials = {'Pt': 'Pt_ONCV_PBE-1.2.upf'}
atoms = io.read(extxyz_path, format='extxyz')
ntyp = len(set(atoms.get_chemical_symbols()))
nat = len(atoms)
ecut=70
input_qe = {
            'control': {
                'calculation': 'scf',
                'outdir': './outdir',
                'pseudo_dir': pseudo_dir,
                'tprnfor': True,
                'disk_io': 'none',
                'restart_mode': 'from_scratch',
            },
            'system': {
                'ibrav': 0,
                'ntyp': ntyp,
                'nat': nat,
                'ecutwfc': ecut,
                'ecutrho': 4 * ecut,
                'input_dft': 'PBE',
                'nbnd': 346,
                'occupations': 'smearing', 
                'smearing': 'marzari-vanderbilt',
                'degauss': 0.02,
            },
            'electrons': {
                'mixing_beta': 0.1,
                'electron_maxstep': 200,
                'diagonalization': 'david',
                'conv_thr': 1e-06,
                'mixing_ndim': 8,
            },
        }
    
bulk = read('./../bulk_optimization/pw.out',format='espresso-out', index=-1) 
P = [[2, 0, 0], [0, 2, 0], [0, 0, 2]]
conf = make_supercell(bulk, P)
initial_positions = conf.get_positions()
initial_cell = conf.get_cell()

max_displacement=0.01 # Maximum displacement in angstrom
max_cell_change=0.01  # Maximum fractional change in cell

num_iterations=100

for i in range(num_iterations):
    current_iteration_dir = os.path.join(outdir,f'scf_{str(i)}')

    # --- 2. Create the directory if it doesn't exist ---
    if not os.path.exists(current_iteration_dir):
        os.makedirs(current_iteration_dir, exist_ok=True)
    
    # --- 3. Generate and write the SLURM submission script (qe.sub) ---
    sub_script_path = os.path.join(current_iteration_dir, "qe.sub")

    script_content = f"""#!/bin/bash
#SBATCH --job-name=scf_{i+1}
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=48
#SBATCH --partition=standard
#SBATCH --time=02:00:00
#SBATCH --mail-type=end
#SBATCH --mail-user=ch24d003@smail.iitm.ac.in
    
mpirun -np 96 -mca coll_hcoll_enable 0 $HOME/qe-6.7/bin/pw.x -npool 2 -in pw.in > pw.out
rm -r outdir
"""
    try:
        with open(sub_script_path, "w") as f:
            f.write(script_content)
        # print(f"Successfully created submission script: {sub_script_path}")
    except IOError as e:
        print(f"Error: Could not write submission script {sub_script_path}. {e}")
        sys.exit(1)
    
    positions=np.copy(initial_positions)
    cell=np.copy(initial_cell)

    # Displace each coordinate randomly
    positions += np.random.rand(positions.shape[0],positions.shape[1])*2*max_displacement - max_displacement
    conf.set_positions(positions)

    # Scale each cell component randomly
    cell *= 1-(np.random.rand(cell.shape[0],cell.shape[1])*2*max_cell_change-max_cell_change)
    conf.set_cell(cell,scale_atoms=True)
	# Define the full path for the output file
    output_file_path = os.path.join(current_iteration_dir, 'pw.in')

    kpts = (4, 4, 4)                    # K-point mesh size
    offset = (0, 0, 0)  

    # Write the QE input file using ASE
    write(output_file_path, conf, format='espresso-in', input_data=input_qe, pseudopotentials=pseudopotentials, kpts=kpts,offset=offset)

# --- 4. Create the Master submit_individual_jobs.sh script ---
# This is created in your 'outdir' (the main directory for all 100 trials)
t_script = os.path.join(outdir, "submit_individual_jobs.sh")

with open(t_script, "w") as f:
    f.write("#!/bin/bash\n")
    # Loop through every directory starting with scf_
    f.write("for dir in scf_*; do\n")
    # If it's a directory, enter it, submit the job, and come back
    f.write('  [ -d "$dir" ] && cd "$dir" && [ -f "qe.sub" ] && sbatch qe.sub && cd ..\n')
    f.write("done\n")
    f.write('echo "All 100 jobs submitted!"\n')

# Make it executable
os.chmod(t_script, 0o755)

print(f"Master script created: {t_script}")
print(f"Done! 100 trials generated in {outdir}")

Master script created: ./submit_individual_jobs.sh
Done! 100 trials generated in ./


In [5]:

# =============================================================================
# CONFIGURATION (Match your generation script)
# =============================================================================
BASE_DIR = "./"
EXPECTED_ATOMS = 32  # Since you used a 2x2x2 supercell of Pt (4 atoms * 8)

# =============================================================================
# HELPER: Validate Calculation (The Physics Gatekeeper)
# =============================================================================
def validate_calculation(conf, expected_atoms):
    """
    Checks ALL physical properties required for a valid training point.
    Returns (True, "OK") or (False, Reason).
    """
    # 1. Check Atom Count
    if len(conf) != expected_atoms:
        return False, f"Wrong Atom Count: Found {len(conf)}, Expected {expected_atoms}"

    # 2. Check Energy (SCF convergence check)
    try:
        e = conf.get_potential_energy()
        if e is None: return False, "Missing Energy"
    except:
        return False, "Energy Not Converged/Missing"

    # 3. Check Forces (Crucial for Machine Learning)
    try:
        f = conf.get_forces()
        if f is None: return False, "Missing Forces"
        if f.shape != (len(conf), 3): return False, "Malformed Forces"
    except:
        return False, "Force Error (Check tprnfor=True)"

    # 4. Check Box/Cell (Crucial for Virial/Stress)
    try:
        c = conf.get_cell()
        if np.linalg.det(c) < 1.0: return False, "Invalid/Collapsed Box"
    except:
        return False, "Box Error"

    return True, "OK"

# =============================================================================
# MAIN DIAGNOSTIC LOOP
# =============================================================================
if __name__ == "__main__":
    print("\n" + "="*60)
    print("STARTING DATA INTEGRITY DIAGNOSTIC: INITIAL TRAINING SET")
    print("="*60 + "\n")

    # Performance Counters
    stats = {"GOOD": 0, "MISSING": 0, "EMPTY": 0, "FAIL": 0, "CRASH": 0}
    failed_folders = []

    # Get all scf_ folders and sort them numerically
    folders = [d for d in os.listdir(BASE_DIR) if d.startswith("scf_")]
    folders.sort(key=lambda x: int(x.split('_')[1]))

    print(f"Checking {len(folders)} folders in {BASE_DIR}...\n")

    for folder in folders:
        file_path = os.path.join(BASE_DIR, folder, 'pw.out')

        # 1. Check if file exists
        if not os.path.exists(file_path):
            print(f"  [MISSING] {folder}")
            stats["MISSING"] += 1
            continue

        # 2. Check if file is empty (Job died instantly)
        if os.path.getsize(file_path) == 0:
            print(f"  [EMPTY]   {folder} (0 bytes)")
            stats["EMPTY"] += 1
            continue

        # 3. Check Contents
        try:
            # ASE reads the output; index=-1 ensures we get the last frame
            conf = read(file_path, format='espresso-out', index=-1)
            
            is_valid, message = validate_calculation(conf, EXPECTED_ATOMS)
            
            if is_valid:
                stats["GOOD"] += 1
            else:
                print(f"  [FAIL]    {folder}: {message}")
                stats["FAIL"] += 1
                failed_folders.append(folder)

        except Exception as e:
            # Catch file corruption or incomplete lines
            print(f"  [CRASH]   {folder}: Unreadable File")
            stats["CRASH"] += 1
            failed_folders.append(folder)

    # =============================================================================
    # FINAL SUMMARY REPORT
    # =============================================================================
    print("\n" + "-"*40)
    print("FINAL INTEGRITY SUMMARY")
    print("-"*40)
    print(f"  ✅ SUCCESSFUL FRAMES:  {stats['GOOD']} / {len(folders)}")
    print(f"  ❌ FAILED/INVALID:     {stats['FAIL']}")
    print(f"  ❌ MISSING (NOT RUN):  {stats['MISSING']}")
    print(f"  ❌ EMPTY (CRASHED):    {stats['EMPTY']}")
    print(f"  ❌ PARSER ERRORS:      {stats['CRASH']}")
    print("-"*40)

    if stats["GOOD"] == len(folders):
        print("RESULT: Dataset is CLEAN and ready for training!")
    else:
        print(f"RESULT: Dataset needs cleanup. Check the folders listed above.")
        if failed_folders:
            print(f"Suggested action: Re-run {len(failed_folders)} failed jobs.")
    print("-"*40 + "\n")


STARTING DATA INTEGRITY DIAGNOSTIC: INITIAL TRAINING SET

Checking 100 folders in ./...


----------------------------------------
FINAL INTEGRITY SUMMARY
----------------------------------------
  ✅ SUCCESSFUL FRAMES:  100 / 100
  ❌ FAILED/INVALID:     0
  ❌ MISSING (NOT RUN):  0
  ❌ EMPTY (CRASHED):    0
  ❌ PARSER ERRORS:      0
----------------------------------------
RESULT: Dataset is CLEAN and ready for training!
----------------------------------------



In [6]:
import os
import numpy as np
from ase.io import read

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = "./"
EXPECTED_ATOMS = 32  # 2x2x2 Supercell of Pt
RERUN_SCRIPT_NAME = "submit_reruns.sh"

# =============================================================================
# MAIN RE-RUN GENERATOR
# =============================================================================
if __name__ == "__main__":
    print("\n" + "="*60)
    print("GENERATING RERUN SCRIPT FOR INITIAL TRAINING SET")
    print("="*60 + "\n")

    rerun_script_path = os.path.join(BASE_DIR, RERUN_SCRIPT_NAME)
    total_reruns = 0
    
    # Get all scf_ folders and sort them numerically
    folders = [d for d in os.listdir(BASE_DIR) if d.startswith("scf_")]
    folders.sort(key=lambda x: int(x.split('_')[1]))

    # Open the shell script for writing
    with open(rerun_script_path, "w") as f_sh:
        f_sh.write("#!/bin/bash\n")
        f_sh.write(f"# Auto-generated rerun script\n")
        f_sh.write("echo 'Starting reruns for failed/missing DFT jobs...'\n\n")

        for folder in folders:
            folder_path = os.path.join(BASE_DIR, folder)
            file_path = os.path.join(folder_path, 'pw.out')
            
            needs_rerun = False
            reason = ""

            # 1. Basic File System Checks
            if not os.path.exists(file_path):
                needs_rerun = True
                reason = "Missing pw.out"
            elif os.path.getsize(file_path) == 0:
                needs_rerun = True
                reason = "Empty pw.out (Immediate Crash)"
            else:
                # 2. Deep Physical Integrity Audit
                try:
                    # Read the last frame
                    conf = read(file_path, format='espresso-out', index=-1)
                    
                    # Audit: Atom Count
                    if len(conf) != EXPECTED_ATOMS:
                        needs_rerun = True
                        reason = f"Atom count mismatch ({len(conf)} vs {EXPECTED_ATOMS})"
                    
                    # Audit: Energy Convergence
                    elif conf.get_potential_energy() is None:
                        needs_rerun = True
                        reason = "Energy calculation failed/incomplete"
                        
                    # Audit: Force Extraction
                    elif conf.get_forces() is None:
                        needs_rerun = True
                        reason = "Missing Forces (Check tprnfor)"

                except Exception as e:
                    needs_rerun = True
                    reason = "Unreadable/Corrupt File"

            # If audits failed, write the Resubmission command to the .sh file
            if needs_rerun:
                print(f"  [RERUN] {folder}: {reason}")
                # We use -D to ensure Slurm runs the job INSIDE the correct folder
                f_sh.write(f"# {folder} failure reason: {reason}\n")
                f_sh.write(f"sbatch -D \"{folder_path}\" \"{folder_path}/qe.sub\"\n")
                total_reruns += 1

        f_sh.write("\necho 'All identified failures have been re-submitted.'\n")
        f_sh.write("echo 'Use squeue -u $USER to monitor progress.'\n")

    # Grant execution permissions to the script (chmod +x)
    os.chmod(rerun_script_path, 0o755)

    # Final Summary
    print("\n" + "-"*40)
    if total_reruns > 0:
        print(f"RERUN SCRIPT CREATED: {rerun_script_path}")
        print(f"TOTAL JOBS TO RE-SUBMIT: {total_reruns}")
        print("-"*40)
        print(f"TO EXECUTE RERUNS, RUN: ./{RERUN_SCRIPT_NAME}")
    else:
        print("ALL JOBS PASSED INTEGRITY CHECK!")
        print("No rerun script needed.")
    print("-"*40 + "\n")


GENERATING RERUN SCRIPT FOR INITIAL TRAINING SET


----------------------------------------
ALL JOBS PASSED INTEGRITY CHECK!
No rerun script needed.
----------------------------------------



In [23]:
# Creating All Directories

snum = input("Enter the system considered: ")
system = input("Enter the name for different system considered for Active learning separated by space: ").split()
graph_num = int(input("Enter the number of DP models considered for training: "))

base_dir = f"./../ActiveLearning_{snum}"

# Creating the main 'DP_models' directory
dp_model_path = os.path.join(base_dir, "DP_models")
if os.path.exists(dp_model_path):
    print(f"Directory already exists: {dp_model_path}")
else:
    os.makedirs(dp_model_path, exist_ok=True)
    print(f"Created new directory: {dp_model_path}")

# Creating system-specific directories
for system_name in system:
    # Creating the 'Training' directory for each system
    train_path = os.path.join(base_dir, system_name, "Training")
    if os.path.exists(train_path):
        print(f"Directory already exists: {train_path}")
    else:
        os.makedirs(train_path, exist_ok=True)
        print(f"Created new directory: {train_path}")
        
graph_dirz = os.path.join(base_dir, "DP_models")
for i in range(graph_num):
    # Create the path for the numbered subdirectory
    numbered_subdir = os.path.join(graph_dirz, str(i + 1))
    
    # Ensure the subdirectory exists
    os.makedirs(numbered_subdir, exist_ok=True)
    print(f"Created directory: {numbered_subdir}")

Enter the system considered:  Pt
Enter the name for different system considered for Active learning separated by space:  bulk
Enter the number of DP models considered for training:  3


Created new directory: ./../ActiveLearning_Pt/DP_models
Created new directory: ./../ActiveLearning_Pt/bulk/Training
Created directory: ./../ActiveLearning_Pt/DP_models/1
Created directory: ./../ActiveLearning_Pt/DP_models/2
Created directory: ./../ActiveLearning_Pt/DP_models/3


In [24]:
import numpy as np
import ase.io
from ase.calculators.espresso import Espresso

# Open output files for writing
file_coord = open(f"./../ActiveLearning_{snum}/bulk/Training/coord.raw", "w")     # Coordinates
file_energy = open(f"./../ActiveLearning_{snum}/bulk/Training/energy.raw", "w")   # Potential energy
file_force = open(f"./../ActiveLearning_{snum}/bulk/Training/force.raw", "w")     # Forces
file_box = open(f"./../ActiveLearning_{snum}/bulk/Training/box.raw", "w")         # Cell dimensions
file_type = open(f"./../ActiveLearning_{snum}/bulk/Training/type.raw", "w")       # Atom types

types_written = False

for i in range(100):
    try:
        conf = ase.io.read(f'./scf_' + str(i) + '/pw.out', format='espresso-out')
    except:
        print("Configuration " + str(i) + " could not be read")
    else:
        try:
            conf.get_forces()
        except:
            print("Forces missing from file" + str(i))
        else:
            # Write data to respective output files
            file_coord.write(' '.join(conf.get_positions().flatten().astype('str').tolist()) + '\n')
            file_energy.write(str(conf.get_potential_energy()) + '\n')
            file_force.write(' '.join(conf.get_forces().flatten().astype('str').tolist()) + '\n')
            file_box.write(' '.join(conf.get_cell().flatten().astype('str').tolist()) + '\n')
            
            if not types_written:
                types = np.array(conf.get_chemical_symbols())
                types[types == "Pt"] = "0"
                file_type.write(' '.join(types.tolist()) + '\n')
                types_written = True

# Close output files
file_coord.close()
file_energy.close()
file_force.close()
file_box.close()
file_type.close()

print("All .raw files created")

All .raw files created


In [25]:
# Test/Validation Set Generation from Training set
import random
import shutil
# Loop through each system to process its training data
for system_name in system:
    # Define paths for the current system
    train_path = os.path.join(base_dir, system_name, 'Training')
    test_path = os.path.join(train_path, 'Testing')

    # Create the test directory for the current system
    try:
        os.makedirs(test_path, exist_ok=True)
    except OSError as e:
        print(f"Error creating directory {test_path}: {e}")
        continue  # Skip to the next system if directory creation fails

    # Define the raw file names
    raw_filenames = ['coord.raw', 'energy.raw', 'force.raw', 'box.raw']

    all_data = {}
    num_configurations = 0

    # Read all data from the generated files for the current system
    print(f"\nProcessing system: {system_name}")
    input_dir = train_path
    for filename in raw_filenames:
        filepath = os.path.join(input_dir, filename)
        if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
            print(f"Warning: '{filename}' in {input_dir} is empty or not found. Cannot proceed with sampling for this system.")
            num_configurations = 0 # Ensure no configs are sampled
            break # Exit the inner loop and continue to the next system
        with open(filepath, 'r') as f:
            lines = f.readlines()
            all_data[filename] = lines
            if filename == 'coord.raw':
                num_configurations = len(lines)
    else: # This 'else' block executes if the 'break' statement is not encountered
        # Proceed with sampling only if files were successfully read
        # Generate 20 random indices to extract
        num_to_extract = 20
        if num_configurations < num_to_extract:
            print(f"Warning: Only {num_configurations} configurations available for {system_name}, extracting all of them.")
            random_indices = list(range(num_configurations))
        else:
            random_indices = random.sample(range(num_configurations), num_to_extract)

        # Write the randomly selected configurations to new files in the testing directory
        for filename in raw_filenames:
            output_filepath = os.path.join(test_path, filename)
            with open(output_filepath, 'w') as f:
                for index in random_indices:
                    f.write(all_data[filename][index])

        # Handle the special case of 'type.raw' by copying it directly
        type_raw_path = os.path.join(input_dir, 'type.raw')
        output_type_raw_path = os.path.join(test_path, 'type.raw')
        if os.path.exists(type_raw_path) and os.path.getsize(type_raw_path) > 0:
            shutil.copy(type_raw_path, output_type_raw_path)
        else:
            print(f"Warning: 'type.raw' was empty or not found for {system_name}, skipping copy.")

        print(f"Successfully extracted {len(random_indices)} random configurations for {system_name} to {test_path}")


Processing system: bulk
Successfully extracted 20 random configurations for bulk to ./../ActiveLearning_Pt/bulk/Training/Testing


In [26]:
import os
import shutil
import random
import numpy as np

# The corrected bash script content as a multi-line string.
bash_script_content = r'''#!/bin/bash
# Default value for nline_per_set
nline_per_set=2000

# Override with the first command-line argument if provided
if [[ $# -ge 1 ]]; then
    nline_per_set=$1
fi

# Clean up previous runs
rm -fr set*

if ! [ -f box.raw ]; then
    echo "Error: box.raw not found. Exiting."
    exit 1
fi

echo "nframe is $(cat box.raw | wc -l)"
echo "nline per set is $nline_per_set"

# Split raw data files into smaller, numbered chunks
split box.raw -l $nline_per_set -d -a 3 box.raw.
split coord.raw -l $nline_per_set -d -a 3 coord.raw.

# Conditionally split other raw files if they exist
test -f energy.raw && split energy.raw -l $nline_per_set -d -a 3 energy.raw.
test -f force.raw && split force.raw -l $nline_per_set -d -a 3 force.raw.
test -f virial.raw && split virial.raw -l $nline_per_set -d -a 3 virial.raw.
test -f atom_ener.raw && split atom_ener.raw -l $nline_per_set -d -a 3 atom_ener.raw.
test -f fparam.raw && split fparam.raw -l $nline_per_set -d -a 3 fparam.raw.
test -f dipole.raw && split dipole.raw -l $nline_per_set -d -a 3 dipole.raw.
test -f polarizability.raw && split polarizability.raw -l $nline_per_set -d -a 3 polarizability.raw.
test -f atomic_dipole.raw && split atomic_dipole.raw -l $nline_per_set -d -a 3 atomic_dipole.raw.
test -f atomic_polarizability.raw && split atomic_polarizability.raw -l $nline_per_set -d -a 3 atomic_polarizability.raw.

# Get the number of sets created by counting all box.raw.*** files
nset=$(ls -1 box.raw.* | wc -l)
nset_1=$((nset - 1))
echo "will make $nset sets"

# Loop through each set
for ii in $(seq 0 $nset_1); do
    pi=$(printf "%03d" $ii)
    echo "making set $ii ($pi)"
    # Create directory with a dot and move files
    mkdir "set.$pi"
    mv "box.raw.$pi" "set.$pi/box.raw"
    mv "coord.raw.$pi" "set.$pi/coord.raw"
    test -f "energy.raw.$pi" && mv "energy.raw.$pi" "set.$pi/energy.raw"
    test -f "force.raw.$pi" && mv "force.raw.$pi" "set.$pi/force.raw"
    test -f "virial.raw.$pi" && mv "virial.raw.$pi" "set.$pi/virial.raw"
    test -f "atom_ener.raw.$pi" && mv "atom_ener.raw.$pi" "set.$pi/atom_ener.raw"
    test -f "fparam.raw.$pi" && mv "fparam.raw.$pi" "set.$pi/fparam.raw"
    test -f "atomic_dipole.raw.$pi" && mv "atomic_dipole.raw.$pi" "set.$pi/atomic_dipole.raw"
    test -f "atomic_polarizability.raw.$pi" && mv "atomic_polarizability.raw.$pi" "set.$pi/atomic_polarizability.raw"
    cd "set.$pi"

    # Convert raw files to .npy format using Python and NumPy with multiline code blocks
    python3 <<END
import os
import numpy as np

if os.path.isfile('box.raw'):
    data = np.loadtxt('box.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('box', data)
if os.path.isfile('coord.raw'):
    data = np.loadtxt('coord.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('coord', data)
if os.path.isfile('energy.raw'):
    data = np.loadtxt('energy.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('energy', data)
if os.path.isfile('force.raw'):
    data = np.loadtxt('force.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('force', data)
if os.path.isfile('virial.raw'):
    data = np.loadtxt('virial.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('virial', data)
if os.path.isfile('atom_ener.raw'):
    data = np.loadtxt('atom_ener.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('atom_ener', data)
if os.path.isfile('fparam.raw'):
    data = np.loadtxt('fparam.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('fparam', data)
if os.path.isfile('dipole.raw'):
    data = np.loadtxt('dipole.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('dipole', data)
if os.path.isfile('polarizability.raw'):
    data = np.loadtxt('polarizability.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('polarizability', data)
if os.path.isfile('atomic_dipole.raw'):
    data = np.loadtxt('atomic_dipole.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('atomic_dipole', data)
if os.path.isfile('atomic_polarizability.raw'):
    data = np.loadtxt('atomic_polarizability.raw', ndmin=2)
    data = data.astype(np.float32)
    np.save('atomic_polarizability', data)
END
    rm *.raw
    cd ..
done
echo "done"
'''

# Loop through each system to generate test/validation sets and the bash script
for system_name in system:
    # Define paths for the current system
    train_path = os.path.join(base_dir, system_name, 'Training')
    test_path = os.path.join(train_path, 'Testing')

    filename = "raw_to_set.sh"
    file_permissions = 0o755  # rwxr-xr-x

    # Write the script to the test directory
    file_path_test = os.path.join(test_path, filename)
    with open(file_path_test, "w") as f:
        f.write(bash_script_content)
    os.chmod(file_path_test, file_permissions)
    print(f"Created executable script '{filename}' in {test_path}.")

    # Write the script to the train directory
    file_path_train = os.path.join(train_path, filename)
    with open(file_path_train, "w") as f:
        f.write(bash_script_content)
    os.chmod(file_path_train, file_permissions)
    print(f"Created executable script '{filename}' in {train_path}.")

Created executable script 'raw_to_set.sh' in ./../ActiveLearning_Pt/bulk/Training/Testing.
Created executable script 'raw_to_set.sh' in ./../ActiveLearning_Pt/bulk/Training.


In [27]:
#########################################################################End of Day 01####################################################















import os
import json
import shutil # Added for safe copy if needed, though dict.copy() is used here

# Define a list of seed values, one for each model.
seed_values = [1, 6534653463541, 652465462461]

# --- Dynamically generate the type_map based on an example file ---
try:
    # Example placeholder logic as per your snippet
    nsym_list = ["Pt"] 
    print(f"Dynamically generated chemical symbols (type_map): {nsym_list}")
except Exception as e:
    print(f"Error reading example file for type_map generation: {e}")
    print("Defaulting to hardcoded type_map.")
    nsym_list = ["Pt"]

# --- Generate the lists of training and validation paths ---
training_set_paths = [os.path.join(base_dir, sys_name, 'Training') for sys_name in system]
validation_set_paths = [os.path.join(base_dir, sys_name, 'Training', 'Testing') for sys_name in system]

# --- Define the base content of the input.json file ---
base_input_json_data = {
    "_comment": "that's all",
    "model": {
        "type_map": nsym_list,
        "descriptor": {
            "type": "se_e2_a",
            "sel": [64],
            "rcut_smth": 3.0,
            "rcut": 6.0,
            "neuron": [25, 50, 100],
            "resnet_dt": False,
            "axis_neuron": 12,
            "seed": None,
            "_comment": " that's all"
        },
        "fitting_net": {
            "neuron": [120, 120, 120],
            "resnet_dt": True,
            "seed": None,
            "_comment": " that's all"
        },
        "_comment": " that's all"
    },
    "learning_rate": {
        "type": "exp",
        "start_lr": 0.005,
        "decay_steps": 10000,
        "_comment": "that's all",
        "stop_lr": 1.752633312441437e-07
    },
    "loss": {
        "start_pref_e": 0.02,
        "limit_pref_e": 1,
        "start_pref_f": 1000,
        "limit_pref_f": 1,
        "start_pref_v": 0,
        "limit_pref_v": 0,
        "_comment": " that's all"
    },
    "training": {
        "stop_batch": 2000000,
        "seed": None,
        "_comment": "that's all",
        "disp_file": "lcurve.out",
        "disp_freq": 100,
        "numb_test": 10,
        "save_freq": 1000,
        "save_ckpt": "model.ckpt",
        "disp_training": True,
        "time_training": True,
        "profiling": False,
        "profiling_file": "timeline.json",
        "training_data": {
            "systems": training_set_paths,
            "set_prefix": "set",
            "batch_size": "auto"
        },
        "validation_data": {
            "systems": validation_set_paths,
            "set_prefix": "set",
            "batch_size": "auto"
        }
    }
}

# --- Define the content of the train.sub file ---
train_sub_content = f'''#!/bin/bash
#SBATCH --job-name=dp
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --gres=gpu:1
#SBATCH --partition=gpu
#SBATCH --time=24:00:00

python -m deepmd train input.json
'''

# --- Write the updated input.json and train.sub to the DP_models directories ---
graph_dirz = os.path.join(base_dir, "DP_models")
os.makedirs(graph_dirz, exist_ok=True) 

# Loop to create numbered subdirectories and write the files
for i in range(graph_num):
    if i >= len(seed_values):
        print(f"Warning: Not enough seed values provided. Using the last seed value ({seed_values[-1]}) for model {i+1}.")
        current_seed = seed_values[-1]
    else:
        current_seed = seed_values[i]
    
    input_json_data = base_input_json_data.copy()

    input_json_data["model"]["descriptor"]["seed"] = current_seed
    input_json_data["model"]["fitting_net"]["seed"] = current_seed
    input_json_data["training"]["seed"] = current_seed
    
    numbered_subdir = os.path.join(graph_dirz, str(i + 1))
    os.makedirs(numbered_subdir, exist_ok=True)
    # print(f"Created directory: {numbered_subdir}")

    # Write input.json
    input_json_path = os.path.join(numbered_subdir, "input.json")
    with open(input_json_path, "w") as f:
        json.dump(input_json_data, f, indent=4)
    # print(f"Wrote input.json to: {input_json_path}")

    # Write train.sub
    train_sub_path = os.path.join(numbered_subdir, "train.sub")
    with open(train_sub_path, "w") as f:
        f.write(train_sub_content)
    os.chmod(train_sub_path, 0o755)
    # print(f"Wrote train.sub to: {train_sub_path}")

# =============================================================================
# NEW: Create Master Submit Script for DP Models
# Location: .../DP_models/submit_dp_training.sh
# =============================================================================
master_submit_path = os.path.join(graph_dirz, "submit_dp_training.sh")
target_sub_script = "train.sub"

master_content = r"""#!/bin/bash
# MASTER SCRIPT FOR DEEPMD TRAINING
# This script finds all numbered folders (1, 2, 3...) in this directory 
# and submits the train.sub script found inside them.

BASE_DIR="PLACEHOLDER_DIR"
TARGET="PLACEHOLDER_TARGET"

echo "--------------------------------------------------------"
echo "Starting DeepMD Training Submission from: $BASE_DIR"
echo "--------------------------------------------------------"

# Loop through all items in the current directory
for dir in *; do
    # Check if it is a directory (e.g., 1, 2, 3)
    if [ -d "$dir" ]; then
        
        # Check if the target submission script exists inside
        if [ -f "$dir/$TARGET" ]; then
            echo "Entering directory: $dir"
            cd "$dir" || continue
            
            # Submit the job
            sbatch "$TARGET"
            
            # Go back to base directory
            cd ..
        fi
    fi
done

echo "--------------------------------------------------------"
echo "All DeepMD training jobs submitted."
"""

# Replace placeholders with Python variables
master_content = master_content.replace("PLACEHOLDER_DIR", graph_dirz)
master_content = master_content.replace("PLACEHOLDER_TARGET", target_sub_script)

try:
    with open(master_submit_path, "w") as f:
        f.write(master_content)
    # Make executable
    os.chmod(master_submit_path, 0o755)
    print(f"\n--> Successfully created master submit script: {master_submit_path}")
    print("    To run: cd DP_models && ./submit_dp_training.sh")
except IOError as e:
    print(f"Error creating master submit script: {e}")

print("\nScript execution complete.")

Dynamically generated chemical symbols (type_map): ['Pt']

--> Successfully created master submit script: ./../ActiveLearning_Pt/DP_models/submit_dp_training.sh
    To run: cd DP_models && ./submit_dp_training.sh

Script execution complete.
